# Appendix 10.4: Prompt Caching

- [Lesson](#lesson)
- [Exercise](#exercise)
- [Example Playground](#example-playground)

## Setup

Run the following setup cell to load your API key and establish the `chat` helper function.

In [ ]:
!pip install anthropic

import anthropic

%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def chat(messages, system=None):
    return client.messages.create(
        model=MODEL_NAME,
        max_tokens=500,
        temperature=0.0,
        system=system,
        messages=messages,
    )

---

## Lesson

**The problem:** a lot of real prompts carry a big, unchanging payload on *every single call* — a long system prompt, a big tool-definition list, a reference document, the growing history of a multi-turn conversation. Without caching, Claude re-reads and re-processes that entire payload from scratch on every request, and you pay full input-token price for it every time, even though 95% of it hasn't changed since the last call.

**Prompt caching** fixes this: you mark a point in your prompt as a cache breakpoint, and Anthropic caches everything up to that point on its servers. The next request that reuses the same prefix gets a **cache hit** — it skips reprocessing that content and reads it from cache instead, which is both **significantly cheaper** (cache reads cost a fraction of normal input tokens) and **noticeably faster** (lower time-to-first-token). Cache entries default to a 5-minute idle TTL (refreshed on each hit), with an extended 1-hour option for longer-lived workloads.

**Where this shows up in real systems:**
- A customer support agent with a 3,000-token system prompt and tool list, answering thousands of tickets a day — without caching you're re-billing yourself for that same system prompt on every single ticket.
- A coding agent (like Claude Code) that keeps a large codebase or file tree in context across many turns of a session — every follow-up question would otherwise reprocess the whole codebase from scratch.
- A chatbot with long conversation history: each new turn resends the entire prior conversation, and without caching that cost grows linearly, turn after turn, for the whole session.
- **The 2 AM production bug this prevents:** a team ships an agent with a large embedded knowledge base in the system prompt, traffic scales up, and the bill spikes 10x overnight because every request reprocesses the same multi-thousand-token block. Caching that one static block is often the single highest-leverage cost fix available.

**Version note:** prompt caching shipped in 2024 as a beta header and has since become a stable, first-class part of the Messages API, with more granular cache breakpoints (up to 4 per request) and the 1-hour cache duration option added since.

### How to use it

Add a `cache_control` field to the content block you want cached — most commonly the last block of a large, static `system` prompt, or the last block of a large static document early in your `messages`. Everything *before and including* that block gets cached as one unit.

```python
system=[
    {
        "type": "text",
        "text": LARGE_STATIC_SYSTEM_PROMPT,
        "cache_control": {"type": "ephemeral"}
    }
]
```

Note there's a **minimum cacheable size** (roughly 1,024-2,048 tokens depending on the model — check current docs for the exact number), so caching only pays off for genuinely large, reused blocks — not a short system prompt.

Let's see it in action. First, we build a large, static "knowledge base" — the kind of thing you'd realistically want cached.

In [ ]:
SECTION = """Section {n}: Return & Refund Policy for Product Line {n}
Customers may request a return within 30 days of delivery for a full refund, provided the item is unused and in
its original packaging. Refunds are issued to the original payment method within 5-7 business days of us
receiving the returned item. Shipping costs for the original order are non-refundable unless the return is due
to a defect or a fulfillment error on our part. Exchanges follow the same 30-day window and are processed as a
return plus a new order once the returned item is received and inspected. Items marked 'final sale' at the time
of purchase are not eligible for return or exchange under any circumstances.
"""

# Repeat to build a long, realistic-sized static knowledge base (well above the caching minimum)
KNOWLEDGE_BASE = "\n".join(SECTION.format(n=i) for i in range(1, 40))
print(f"Knowledge base is roughly {len(KNOWLEDGE_BASE.split())} words")

Now let's call Claude twice with this knowledge base as a cached system prompt, and inspect `response.usage` to see the cache in action. The first call **writes** to the cache; the second call should **read** from it.

In [ ]:
system = [
    {
        "type": "text",
        "text": "You are a support assistant. Answer questions using only the policy knowledge base below.\n\n" + KNOWLEDGE_BASE,
        "cache_control": {"type": "ephemeral"}
    }
]

response1 = chat([{"role": "user", "content": "What's the return window for Product Line 5?"}], system=system)
print("Call 1 usage:", response1.usage)

response2 = chat([{"role": "user", "content": "What's the return window for Product Line 12?"}], system=system)
print("Call 2 usage:", response2.usage)

On the first call, watch for `cache_creation_input_tokens` — that's the (slightly more expensive) one-time cost of writing the cache. On the second call, that same content should show up as `cache_read_input_tokens` instead — read from cache, billed at a fraction of the normal input rate, with lower latency. Everything *after* the cached block (the actual user question) is still processed fresh each time, as it should be, since it changes every call.

---

## Exercise

### Exercise 10.4.1 - Cache the Right Thing

Below is a prompt setup for a code review bot. It has three parts: a large static **style guide** (the same every time), a **file the reviewer is looking at** (changes per request, but is often large), and a short **user question** (always changes). Add a `cache_control` breakpoint in the right place so that the style guide is cached and reused across requests, without caching the parts that change every call.

In [ ]:
STYLE_GUIDE = ("Our Python style guide: use type hints on all public functions; prefer f-strings; "
               "functions should do one thing; no bare except clauses; docstrings only when the 'why' "
               "isn't obvious from the name. " * 200)  # padded to realistic length

# System prompt - this is the only field you should change
system = """[Replace this text with a system list that puts STYLE_GUIDE in its own content block
with a cache_control breakpoint]"""

file_under_review = "def add(a,b):\n  return a+b"

messages = [{"role": "user", "content": f"Review this file for style issues:\n{file_under_review}"}]

response = chat(messages, system=system)
print(response.content[0].text)
print("\n--------------------------- GRADING ---------------------------")
print("cache_creation_input_tokens on this call:", response.usage.cache_creation_input_tokens)

❓ If you want a hint, run the cell below!

In [ ]:
from hints import exercise_10_4_1_hint; print(exercise_10_4_1_hint)

### Congrats!

**Recap:** prompt caching lets you mark a large, reused prefix of your prompt (system instructions, tool definitions, reference documents, growing conversation history) so Claude skips reprocessing it on later calls — cheaper and faster, with no change to what Claude actually sees or how it answers.

**Interview-answer framing:** "Prompt caching addresses the fact that a lot of production prompts resend the same large static content on every call — a system prompt, tool schema, or reference document. By marking a `cache_control` breakpoint, that prefix gets cached server-side for a few minutes (or up to an hour with the extended option), so repeat calls read it from cache instead of reprocessing it — cutting cost and latency substantially for high-traffic or long-context workloads, without changing the model's behavior at all."

Head over to Appendix 10.5 to learn about extended thinking.

---

## Example Playground

This is an area for you to experiment freely with the prompt examples shown in this lesson.

In [ ]:
SECTION = """Section {n}: Return & Refund Policy for Product Line {n}
Customers may request a return within 30 days of delivery for a full refund, provided the item is unused and in
its original packaging. Refunds are issued to the original payment method within 5-7 business days of us
receiving the returned item.
"""
KNOWLEDGE_BASE = "\n".join(SECTION.format(n=i) for i in range(1, 40))

system = [
    {
        "type": "text",
        "text": "You are a support assistant. Answer questions using only the policy knowledge base below.\n\n" + KNOWLEDGE_BASE,
        "cache_control": {"type": "ephemeral"}
    }
]

response = chat([{"role": "user", "content": "What's the return window for Product Line 5?"}], system=system)
print(response.content[0].text)
print(response.usage)